# Cleaning and Feature Audit

This notebook cleans the raw job data, applies rule-based classification and feature engineering, and checks data quality at each step.

Workflow:
1. Read and combine multiple batches of raw job data;
2. Review cleaned row counts, data-quality issues, and classification distributions;
3. Filter Canadian early-career jobs for the main analysis;
4. Validate the engineered features and export the processed data.

## 1. Import dependencies

The functions below handle data cleaning, feature engineering, and review-queue construction. Paths are loaded from the project configuration rather than hard-coded in the notebook.

In [1]:
import sys
sys.path.append("..")
from src.clean import clean_jobs
from src.feature_engineering import engineer_features
import pandas as pd
from IPython.display import display
from config.settings import RAW_DATA_DIR, PROJECT_ROOT
from src.review import build_review_queue
from src.pipeline import _select_analysis_jobs

## 2. Load and clean raw data

The four JSONL files represent separate collection batches. They are combined into one raw DataFrame, then passed to `clean_jobs` to handle duplicate records, missing values, and invalid fields. The issue lists are retained for auditing.

In [2]:
# Read data
df_1 = pd.read_json(RAW_DATA_DIR / 'jobs_20260911_071658.jsonl', lines=True)
df_2 = pd.read_json(RAW_DATA_DIR / 'jobs_20260920_005742.jsonl', lines=True)
df_3 = pd.read_json(RAW_DATA_DIR / 'jobs_20260920_011920.jsonl', lines=True)
df_4 = pd.read_json(RAW_DATA_DIR / 'jobs_20260920_012618.jsonl', lines=True)
df_raw = pd.concat([df_1, df_2, df_3, df_4],ignore_index=True)
df_clean, id_issues, search_history, employment_type_issues, location_issues = clean_jobs(df_raw)
df_features = engineer_features(df_clean)
review_queue = build_review_queue(df_features)

classified_role_family review labels need manual sampling: 'other' (76), 'hybrid' (7), 'adjacent_hybrid' (1)


classified_career_level review labels need manual sampling: 'unknown' (161), 'potential_senior' (59)


## 3. Audit cleaning and classification results

First compare the number of raw, cleaned, and feature-engineered records, then review field-level quality flags. The main checks are whether cleaning removed an unexpected number of records, whether invalid fields were identified, and whether the automatic classifications have a reasonable distribution.

In [3]:
# Existing number of rows.
counts = pd.Series({
    'raw_rows': len(df_raw),
    'cleaned_rows': len(df_clean),
    'missing_ids': len(id_issues),
    'employment_type_issues': len(employment_type_issues),
    'location_issues': len(location_issues),
    'featured_rows': len(df_features),
    'review_queue': len(review_queue),
}).rename_axis('name').reset_index(name='count')
display(counts)

,name,count
0,raw_rows,643
1,cleaned_rows,492
2,missing_ids,0
3,employment_type_issues,8
4,location_issues,18
5,featured_rows,492
6,review_queue,282


In [4]:
# Count records marked by quality flags.
flag_columns = [
    'is_senior',
    'is_future_posting',
    'has_invalid_posting_date',
    'is_short_description',
    'is_missing_description',
]
display(df_features[flag_columns].fillna(False).sum().rename_axis('flag').reset_index(name='count'))

,flag,count
0,is_senior,59
1,is_future_posting,0
2,has_invalid_posting_date,11
3,is_short_description,34
4,is_missing_description,0


In [5]:
# Review role-family and career-level distributions.
display(df_features['classified_role_family'].value_counts().rename_axis('classified_role_family').reset_index(name='count'))
display(df_features['classified_career_level'].value_counts().rename_axis('classified_career_level').reset_index(name='count'))
display(pd.crosstab(df_features['classified_role_family'], df_features['classified_career_level']))

,classified_role_family,count
0,business_analyst,98
1,data_scientist,96
2,data_analyst,87
3,other,76
4,data_engineer,72
5,bi_analyst,30
6,outside_scope,25
7,hybrid,7
8,adjacent_hybrid,1


,classified_career_level,count
0,internship,205
1,unknown,161
2,potential_senior,59
3,entry_level,46
4,new_grad,21


classified_career_level,entry_level,internship,new_grad,potential_senior,unknown
classified_role_family,,,,,
adjacent_hybrid,1,0,0,0,0
bi_analyst,1,8,0,5,16
business_analyst,3,41,2,14,38
data_analyst,11,42,9,4,21
data_engineer,9,33,2,9,19
data_scientist,7,33,1,16,39
hybrid,0,0,2,4,1
other,6,41,4,5,20
outside_scope,8,7,1,2,7


## 4. Build the main analysis sample

The main analysis sample keeps target roles at the internship, new-graduate, or entry-level career stages in Canada, provided that their date and location fields pass quality checks. Records marked as senior, old or future postings, invalid dates, or invalid locations are excluded. The complete cleaned dataset remains available in `df_features`.

In [6]:
# Build the filtered sample for the main analysis.
df_analysis = _select_analysis_jobs(df_features)

print('Rows of df_features:', len(df_features))
print('Rows of df_analysis:', len(df_analysis))


Rows of df_features: 492
Rows of df_analysis: 180


In [7]:
# Classification and distribution for df_analysis
display(pd.crosstab(
    df_analysis['classified_role_family'],
    df_analysis['classified_career_level'],
    margins=True,
))

classified_career_level,entry_level,internship,new_grad,All
classified_role_family,,,,
bi_analyst,1,7,0,8
business_analyst,2,41,1,44
data_analyst,11,42,8,61
data_engineer,7,27,1,35
data_scientist,3,28,1,32
All,24,145,11,180


In [8]:
# Sample survey
sample = df_analysis.sample(n=min(10, len(df_analysis)), random_state=42)

display(sample[[
    'job_title',
    'classified_role_family',
    'classified_career_level',
    'career_keyword',
    'source',
]])

,job_title,classified_role_family,classified_career_level,career_keyword,source
66,Business System Analyst Intern/Co-Op (Winter 2...,business_analyst,internship,intern,job_title
162,Business Analyst,business_analyst,internship,stage,job_description
446,Junior Data Analyst Intern/Co-op,data_analyst,internship,intern,job_title
320,GRO - Data Analyst,data_analyst,new_grad,new graduate,job_description
430,"Insights & Integrations Consultancy, Data Scie...",data_scientist,internship,intern,job_title
61,Business System Analyst Co-Op / Intern (Winter...,business_analyst,internship,intern,job_title
88,SAP Business Analyst - Finance (SAP FI/CO) (Bi...,business_analyst,internship,stage,job_description
275,New Graduate Program - 2027 Data Science Assoc...,data_scientist,new_grad,new graduate,job_title
379,Junior Data Engineer,data_engineer,entry_level,junior,job_title
385,Corporate HR Data Analyst Internship,data_analyst,internship,internship,job_title


### Sample definition and limitations

This analysis is limited to records posted within the last 35 days, with the country field set to Canada, and identified by the automatic rules as target roles. Career stages include `internship`, `new_grad`, and `entry_level`. Records suspected to be senior, with invalid dates, old or future posting dates, or invalid locations are excluded. Categories such as `unknown` remain in the complete dataset but are not included in the main analysis. The automatic classification has received only limited sample-based review, so misclassifications and omissions may remain.

## 5. Validate engineered features

Feature engineering and validation are applied to the complete cleaned dataset above. The main analysis sample is a filtered view of those featured records. Text-extraction outputs can be reviewed using random samples, and feature coverage is calculated for the complete featured dataset.

In [9]:
# Check the structure of the data
featured_df = df_features
print('featured_df shape:', featured_df.shape)
print('featured_df columns:', featured_df.columns.tolist())
print('featured_df dtypes:', featured_df.dtypes)

featured_df shape: (492, 47)
featured_df columns: ['job_id', 'job_title', 'employer_name', 'job_state', 'job_country', 'job_city', 'job_latitude', 'job_longitude', 'job_employment_type', 'job_is_remote', 'job_posted_at_datetime_utc', 'job_description', 'job_apply_link', 'job_min_salary', 'job_max_salary', 'job_salary_currency', 'job_salary_period', 'role_family', 'role_query', 'search_level', 'search_query', 'page_number', 'collected_at_utc', 'job_province_clean', 'job_city_clean', 'location_status', 'market_area', 'description_length', 'is_short_description', 'is_missing_description', 'posting_age_days', 'is_old_posting', 'is_future_posting', 'has_invalid_posting_date', 'classified_role_family', 'role_status', 'is_senior', 'classified_career_level', 'career_keyword', 'source', 'career_level_status', 'accepted_majors', 'education_text', 'technical_skills', 'soft_skills', 'language_requirements', 'stated_experience_years']
featured_df dtypes: job_id                                     s

In [10]:
# Check that feature engineering does not add or delete rows unexpectedly.
assert len(featured_df) == len(df_features)
# Check the job_id is unique
assert featured_df['job_id'].is_unique
# Check the head 5 row in each core columns in featured_df
FEATURE_COLUMNS = [
    'classified_role_family',
    'classified_career_level',
    'career_level_status',
    'is_senior',
    'technical_skills',
    'soft_skills',
    'language_requirements',
    'accepted_majors',
    'stated_experience_years',
]
for col in FEATURE_COLUMNS:
    print(f"Head 5 rows of column '{col}':")
    display(featured_df[[col]].head())

Head 5 rows of column 'classified_role_family':


,classified_role_family
0,outside_scope
1,data_analyst
2,data_scientist
3,data_analyst
4,business_analyst


Head 5 rows of column 'classified_career_level':


,classified_career_level
0,internship
1,internship
2,internship
3,internship
4,internship


Head 5 rows of column 'career_level_status':


...

,language_requirements
0,"[english, french, bilingual]"
1,[]
2,[]
3,[]
4,[]


Head 5 rows of column 'accepted_majors':


,accepted_majors
0,[]
1,"[engineering, economics]"
2,[]
3,[]
4,[]


Head 5 rows of column 'stated_experience_years':


,stated_experience_years
0,0
1,<NA>
2,5
3,<NA>
4,3


In [11]:
# Sample validation text extraction
display(featured_df[['job_description', 'technical_skills']].sample(20, random_state=42))
display(featured_df[['job_description', 'soft_skills']].sample(20, random_state=42))
display(featured_df[['job_description', 'language_requirements']].sample(20, random_state=42))
display(featured_df[['education_text', 'accepted_majors']].sample(20, random_state=42))
display(featured_df[['job_description', 'stated_experience_years']].sample(20, random_state=42))

,job_description,technical_skills
78,We Are Banking At Another Level.\n\r\n\n\r\nCh...,"[sql, excel, power_bi, azure, databricks]"
55,BMO Capital Markets\n\r\n\n\r\nBMO Capital Mar...,[]
272,Manufacturing Design Engineer\n\r\n\n\r\nJob T...,[]
286,"At IBM Consulting FutureNow, we unite human ex...","[python, sql, power_bi, azure, databricks]"
63,RBC in Ontario is offering a 4-month Winter 20...,[]
182,Role Type\n\r\n\n\r\nInternship/Co-op\n\r\n\n\...,"[python, sql, pandas, numpy, scikit_learn, spa..."
30,Not the Flashiest Title in the Building. Possi...,"[python, sql]"
117,Job Posting: Financial Business Analyst\n\r\nA...,[excel]
464,"Embark on a data-driven journey with Ericsson,...","[python, aws]"
336,Introduction A career in IBM Consulting is roo...,"[sql, excel, tableau, power_bi, azure]"


,job_description,soft_skills
78,We Are Banking At Another Level.\n\r\n\n\r\nCh...,"[communication, teamwork, attention_to_detail]"
55,BMO Capital Markets\n\r\n\n\r\nBMO Capital Mar...,[]
272,Manufacturing Design Engineer\n\r\n\n\r\nJob T...,"[communication, attention_to_detail]"
286,"At IBM Consulting FutureNow, we unite human ex...",[problem_solving]
63,RBC in Ontario is offering a 4-month Winter 20...,[]
182,Role Type\n\r\n\n\r\nInternship/Co-op\n\r\n\n\...,[]
30,Not the Flashiest Title in the Building. Possi...,[attention_to_detail]
117,Job Posting: Financial Business Analyst\n\r\nA...,"[communication, teamwork, attention_to_detail]"
464,"Embark on a data-driven journey with Ericsson,...",[problem_solving]
336,Introduction A career in IBM Consulting is roo...,"[communication, teamwork, problem_solving, att..."


,job_description,language_requirements
78,We Are Banking At Another Level.\n\r\n\n\r\nCh...,"[english, french, bilingual]"
55,BMO Capital Markets\n\r\n\n\r\nBMO Capital Mar...,"[english, french, bilingual]"
272,Manufacturing Design Engineer\n\r\n\n\r\nJob T...,[]
286,"At IBM Consulting FutureNow, we unite human ex...","[english, french, bilingual]"
63,RBC in Ontario is offering a 4-month Winter 20...,[]
182,Role Type\n\r\n\n\r\nInternship/Co-op\n\r\n\n\...,[]
30,Not the Flashiest Title in the Building. Possi...,[]
117,Job Posting: Financial Business Analyst\n\r\nA...,"[english, french, bilingual]"
464,"Embark on a data-driven journey with Ericsson,...",[]
336,Introduction A career in IBM Consulting is roo...,[]


,education_text,accepted_majors
78,- Currently enrolled in a university program i...,"[computer_science, mathematics, engineering, b..."
55,- Currently enrolled in an undergraduate degre...,[]
272,,[]
286,- University or college degree in computer sci...,"[computer_science, data_science, engineering]"
63,,[]
182,- Must be enrolled in an undergraduate/ gradua...,[]
30,,[]
117,"- Bachelor's degree in Accounting, Finance, or...","[business, finance]"
464,• Enrolled in a relevant undergraduate or post...,[]
336,All applicants must be eligible to work full-...,[business]


,job_description,stated_experience_years
78,We Are Banking At Another Level.\n\r\n\n\r\nCh...,<NA>
55,BMO Capital Markets\n\r\n\n\r\nBMO Capital Mar...,<NA>
272,Manufacturing Design Engineer\n\r\n\n\r\nJob T...,2
286,"At IBM Consulting FutureNow, we unite human ex...",1
63,RBC in Ontario is offering a 4-month Winter 20...,<NA>
182,Role Type\n\r\n\n\r\nInternship/Co-op\n\r\n\n\...,<NA>
30,Not the Flashiest Title in the Building. Possi...,<NA>
117,Job Posting: Financial Business Analyst\n\r\nA...,0
464,"Embark on a data-driven journey with Ericsson,...",<NA>
336,Introduction A career in IBM Consulting is roo...,<NA>


In [12]:
# Check the abnormal stated_experience_years values
display(featured_df.loc[featured_df['stated_experience_years'] > 10, ['job_description', 'stated_experience_years']])

,job_description,stated_experience_years
71,*English version below*\n\r\n\n\r\n\n\r\nNous ...,30
114,Position Description:\n\r\nContribute to the s...,35
164,Temps-Plein / Full-Time\n\r\n\n\r\nMode de tra...,35
234,Robots & Pencils is an applied AI engineering ...,15


In [13]:
# Check the coverage of each feature
feature_coverage = pd.Series({
    'technical_skills': featured_df['technical_skills'].str.len().gt(0).mean(),
    'soft_skills': featured_df['soft_skills'].str.len().gt(0).mean(),
    'language_requirements': featured_df['language_requirements'].str.len().gt(0).mean(),
    'accepted_majors': featured_df['accepted_majors'].str.len().gt(0).mean(),
    'stated_experience_years': featured_df['stated_experience_years'].notna().mean(),
}).rename_axis('feature').reset_index(name='coverage')
display(feature_coverage)

,feature,coverage
0,technical_skills,0.701220
1,soft_skills,0.760163
2,language_requirements,0.310976
3,accepted_majors,0.479675
4,stated_experience_years,0.288618


## 6. Export results

The complete featured dataset and filtered analysis sample are written separately to `data/processed/featured_jobs.jsonl` and `data/processed/analysis_jobs.jsonl`, matching the pipeline outputs.

In [14]:
# Save the full featured data and filtered analysis sample separately.
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
featured_df.to_json(processed_dir / 'featured_jobs.jsonl', orient='records', lines=True, force_ascii=False, date_format='iso')
df_analysis.to_json(processed_dir / 'analysis_jobs.jsonl', orient='records', lines=True, force_ascii=False, date_format='iso')